# Task 1.1

Eksempel på en robust filleser i Python.

In [ ]:
from pathlib import Path


def read_file_robust(path):
    """Read a text file using UTF-8 or ISO-8859-1/Latin-1.

    Returns:
        tuple[str, str]: (file_text_content, encoding_used)
    """
    file_path = Path(path)
    try:
        text = file_path.read_text(encoding='utf-8')
        return text, 'utf-8'
    except UnicodeDecodeError:
        text = file_path.read_text(encoding='iso-8859-1')
        return text, 'iso-8859-1'
    except FileNotFoundError:
        raise

# Eksempelbruk:
# content, encoding = read_file_robust('example.txt')
# print(content)
# print(encoding)


#4.1
import pandas as pd

def create_sales_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """
    Konverterer transaksjonslisten til en Pandas DataFrame, setter timestamp 
    som indeks (konvertert til datetime), og sorterer kronologisk.
    """
    df = pd.DataFrame(transactions)
    
    # Konverter timestamp til datetime og sett som indeks
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.set_index('timestamp', inplace=True)
    
    # Sorter DataFrame kronologisk etter indeks (tid)
    df.sort_index(inplace=True)
    
    return df

#4.2
def resample_sales_by_item(df: pd.DataFrame) -> pd.DataFrame:
    """
    Grupperer data per item og resampler i 1-timers vinduer med summen av prisen.
    """
    resampled_df = df.groupby('item').resample('1h')['price'].sum()
    
    return resampled_df

#4.3
def create_revenue_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Oppretter en pivot-tabell som viser total transaksjonsomsetning (sum av pris)
    gruppert etter item på tvers av statuskategoriene.
    """
    pivot_table = pd.pivot_table(
        df,
        values='price',
        index='item',
        columns='status',
        aggfunc='sum',
        fill_value=0.0
    )
    
    return pivot_table

#4.4
def calculate_refund_ratio_from_pivot(df: pd.DataFrame) -> pd.Series:
    """
    Beregner refusjonsraten per item ved å kalle funksjonen fra oppgave 3 (pivot-tabellen).
    """
    pivot = create_revenue_pivot_table(df)
    
    refunds = pivot['REFUND'] if 'REFUND' in pivot.columns else 0
    success = pivot['SUCCESS'] if 'SUCCESS' in pivot.columns else 0
    
    total_activity = pivot.sum(axis=1)
    refund_ratio = refunds / total_activity.replace(0, 1)
    
    return refund_ratio
